#### ENRICH `ORDER_PAYMENTS` FROM BRONZE TO SILVER

In [0]:
CATALOG = 'ORDERS_SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'ORDER_PAYMENTS_ENRICHED'

In [0]:
order_payments_df = spark.table('ORDERS_SALES.BRONZE.ORDERS_PAYMENTS')
order_payments_df.limit(1).display()

#### INCREMENTAL LOAD `ORDERS_SALES.SILVER.ORDER_PAYMENTS_ENRICHED`

In [0]:
spark.sql(f'''MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_order_items_temp_vw
  )
  WHERE rn = 1
) src
ON (src.order_id = tgt.order_id)
WHEN MATCHED
    THEN UPDATE SET
    tgt.order_item_id = src.order_item_id,
    tgt.product_id = src.product_id,
    tgt.seller_id = src.seller_id,
    tgt.shipping_limit_date = src.shipping_limit_date
WHEN NOT MATCHED THEN
INSERT (order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value, is_price_non_positive,
is_freight_negative, is_ship_date_outlier, total_item_value, freight_ratio, ship_year, ship_month, ship_date, origin_file_path,
load_timestamp)
VALUES (src.order_id, src.order_item_id, src.product_id, src.seller_id, src.shipping_limit_date, src.price, src.freight_value,
src.is_price_non_positive, src.is_freight_negative, src.is_ship_date_outlier, src.total_item_value, src.freight_ratio, 
src.ship_year, src.ship_month, src.ship_date, src.file_path, current_timestamp)
''')

In [0]:
dbutils.notebook.exit('SUCCESSFULLY LOADED INTO ORDERS_SALES.SILVER.ORDER_ITEMS_ENRICHED')